In [ ]:
import json
from tqdm import tqdm
from openai import OpenAI

# Create OpenAI client with SSL verification disabled
# API

import httpx
client = OpenAI(
    api_key=api_key,
    http_client=httpx.Client(verify=False)  # Disable SSL verification safely
)

with open('valid_data.json', 'r') as vd:
    valid_json = json.load(vd)

# Collect results from LLM in dictionary
results = {}

# Use tqdm for progress bar
for entry in tqdm(valid_json, desc="Processing entries"):
    natural_s = valid_json[entry]["natural_s"]
    natural_p = valid_json[entry]["natural_p"]
    formal_s = valid_json[entry]["formal_s"]

    # Call GPT to perform formal_s to natural language
    
    prompt = f"""
            Translate the following Isabelle/HOL statement into a clear natural language question that conveys the exact same logical meaning.
            
            Generate only the natural language question written in LaTex, without any additional commentary or explanation.

            Formal Statement:
            {formal_s}

            Natural Language Statement:
            """
    
    # Call GPT
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that translates between formal logic and natural language."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=500,
        n=1,
        stop=None,
        temperature=0.0,
    )

    # Collect Response
    gpt_natural_s = response.choices[0].message.content

    results[entry] = {
        "natural_s": natural_s,
        "natural_p": natural_p,
        "formal_s": formal_s,
        "gpt_natural_s": gpt_natural_s
    }

# Save results to a JSON file
with open('gpt_translations_latex.json', 'w') as out_file:
    json.dump(results, out_file, indent=4)

Processing entries: 100%|██████████| 244/244 [05:59<00:00,  1.47s/it]


In [ ]:
import json
from tqdm import tqdm
from openai import OpenAI

# Create OpenAI client with SSL verification disabled
# API

import httpx
client = OpenAI(
    api_key=api_key,
    http_client=httpx.Client(verify=False)  # Disable SSL verification safely
)

with open('valid_data.json', 'r') as vd:
    valid_json = json.load(vd)

# Collect results from LLM in dictionary
results = {}

# Use tqdm for progress bar
for entry in tqdm(valid_json, desc="Processing entries"):
    natural_s = valid_json[entry]["natural_s"]
    natural_p = valid_json[entry]["natural_p"]
    formal_s = valid_json[entry]["formal_s"]

    # Call GPT to perform formal_s to natural language
    
    prompt = f"""
            Translate the following Natural Language statement into a Isabelle/HOL Formal Statement that conveys the exact same logical meaning.
                
            Generate only the Isabelle/HOL Formal Statement, without any additional commentary or explanation.

            Natural Language Statement:
            {natural_s}

            Formal Statement:
            """
    
    # Call GPT
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that translates between formal logic and natural language."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=500,
        n=1,
        stop=None,
        temperature=0.0,
    )

    # Collect Response
    gpt_natural_s = response.choices[0].message.content

    results[entry] = {
        "natural_s": natural_s,
        "natural_p": natural_p,
        "formal_s": formal_s,
        "gpt_formal_s": gpt_natural_s
    }

# Save results to a JSON file
with open('gpt_translations_latex_1_formal.json', 'w') as out_file:
    json.dump(results, out_file, indent=4)

Processing entries: 100%|██████████| 244/244 [05:49<00:00,  1.43s/it]


In [ ]:
import json
from tqdm import tqdm
from openai import OpenAI

# Create OpenAI client with SSL verification disabled
# API

import httpx
client = OpenAI(
    api_key=api_key,
    http_client=httpx.Client(verify=False)  # Disable SSL verification safely
)

with open('valid_data.json', 'r') as vd:
    valid_json = json.load(vd)

# valid_dic[data['problem_name']]={"natural_s":data["informal_statement"],
#                                  "natural_p":data["informal_proof"], 
#                                  "formal_s":data["formal_statement"]}

# Collect results from LLM in dictionary
results = {}

# Use tqdm for progress bar
for entry in tqdm(valid_json, desc="Processing entries"):
    natural_s = valid_json[entry]["natural_s"]
    natural_p = valid_json[entry]["natural_p"]
    formal_s = valid_json[entry]["formal_s"]

    # Call GPT to perform formal_s to natural language
    
    prompt = f"""
            Translate the following Isabelle/HOL statement into 10 clear natural language questions that conveys the exact same logical meaning.
            
            Generate a numbered list of 10 unique natural language questions that conveys the exact same logical meaning written in LaTex, without any additional commentary or explanation.

            Formal Statement:
            {formal_s}

            Natural Language Statements:
            1.
            """
    
    # Call GPT
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that translates between formal logic and natural language."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=5000,
        n=1,
        stop=None,
        temperature=0.0,
    )

    # Collect Response
    gpt_natural_s = response.choices[0].message.content

    results[entry] = {
        "natural_s": natural_s,
        "natural_p": natural_p,
        "formal_s": formal_s,
        "gpt_natural_s": gpt_natural_s
    }

# Save results to a JSON file
with open('gpt_translations_latex_10.json', 'w') as out_file:
    json.dump(results, out_file, indent=4)

Processing entries: 100%|██████████| 244/244 [24:21<00:00,  5.99s/it]


In [4]:
with open('gpt_translations_latex_10.json', 'r') as out_file:
    my_json = json.load(out_file)


for entry in my_json:
    # Try to split based on "\item" after removing "\begin{enumerate}" and "\end{enumerate}"
    # Otherwise we need to split based on new lines
    if "\\begin{enumerate}" in my_json[entry]["gpt_natural_s"]:
        items = my_json[entry]["gpt_natural_s"].replace("\\begin{enumerate}", "").replace("\\end{enumerate}", "").split("\\item")
        # Remove any leading/trailing whitespace and filter out empty strings
        items = [item.strip() for item in items if item.strip()]
        my_json[entry]["gpt_natural_s_list"] = items
    else:
        items = my_json[entry]["gpt_natural_s"].split("\n")
        # Remove leading numbers after splitting on dot or parenthesis
        items = [item.split('.', 1)[-1] if '.' in item else item.split(')', 1)[-1] for item in items]
        
        items = [item.strip() for item in items if item.strip()]
        my_json[entry]["gpt_natural_s_list"] = items
    
   
with open('gpt_translations_latex_10_formatted.json', 'w') as out_file:
    json.dump(my_json, out_file, indent=4)


In [ ]:
import json
from tqdm import tqdm
from openai import OpenAI

# Create OpenAI client with SSL verification disabled
# API

import httpx
client = OpenAI(
    api_key=api_key,
    http_client=httpx.Client(verify=False)  # Disable SSL verification safely
)

with open('gpt_translations_latex_10_formatted.json', 'r') as vd:
    valid_json = json.load(vd)

# Collect results from LLM in dictionary
results = {}

# Use tqdm for progress bar
for entry in tqdm(valid_json, desc="Processing entries"):
    gpt_natural_s_list = valid_json[entry]["gpt_natural_s_list"]
    gpt_formal_s_list = []
    natural_s = valid_json[entry]["natural_s"]
    natural_p = valid_json[entry]["natural_p"]
    formal_s = valid_json[entry]["formal_s"]
    for i, gpt_natural_s in enumerate(gpt_natural_s_list):  
        # Call GPT to perform formal_s to natural language
        prompt = f"""
                Translate the following Natural Language statement into a Isabelle/HOL Formal Statement that conveys the exact same logical meaning.
                
                Generate only the Isabelle/HOL Formal Statement, without any additional commentary or explanation.

                Natural Language Statement:
                {gpt_natural_s}

                Formal Statement:
                """
        
        # Call GPT
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that translates between formal logic and natural language."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=500,
            n=1,
            stop=None,
            temperature=0.0,
        )

        # Collect Response
        gpt_formal_s = response.choices[0].message.content
        gpt_formal_s_list.append(gpt_formal_s)

    results[entry] = {
        "natural_s": natural_s,
        "natural_p": natural_p,
        "formal_s": formal_s,
        "gpt_natural_s": gpt_natural_s,
        "gpt_natural_s_list": gpt_natural_s_list,
        "gpt_formal_s_list": gpt_formal_s_list 
    }

# Save results to a JSON file
with open('gpt_translations_latex_10_roundtrip.json', 'w') as out_file:
    json.dump(results, out_file, indent=4)

Processing entries: 100%|██████████| 244/244 [44:11<00:00, 10.87s/it]


In [14]:
# parse each entities gpt_formal_s_list and remove  ```isabelle\n AND  \n```
with open('/Users/hmmoore/Desktop/AutoFormalization/Data/gpt_translations_latex_1_formal.json', 'r') as out_file:
    my_json = json.load(out_file)

for entry in my_json:
    formal_s = my_json[entry]["gpt_formal_s"]
    # Remove ```isabelle\n at the start and \n``` at the end if they exist
    if formal_s.startswith("```isabelle"):
        # replace with theorem:\n
        formal_s = formal_s.replace("```isabelle\n", "theorem:\n ").lstrip()
    else:
        # add theorem:\n at the start
        formal_s = "theorem:\n " + formal_s.lstrip()
    if formal_s.endswith("```"):
        formal_s = formal_s[:-len("```")].rstrip()
    my_json[entry]["gpt_formal_s"] = formal_s

# Save cleaned results to a JSON file
with open('/Users/hmmoore/Desktop/AutoFormalization/Data/gpt_translations_latex_1_formal_cleaned.json', 'w') as out_file:
    json.dump(my_json, out_file, indent=4)

In [ ]:
import json
from tqdm import tqdm
from openai import OpenAI

# Create OpenAI client with SSL verification disabled
# API

import httpx
client = OpenAI(
    api_key=api_key,
    http_client=httpx.Client(verify=False)  # Disable SSL verification safely
)

with open('gpt_translations_latex_10_roundtrip_cleaned.json', 'r') as vd:
    valid_json = json.load(vd)

# Collect results from LLM in dictionary
results = {}

# Use tqdm for progress bar
for entry in tqdm(valid_json, desc="Processing entries"):
    natural_s = valid_json[entry]["natural_s"]
    natural_p = valid_json[entry]["natural_p"]
    formal_s = valid_json[entry]["formal_s"]

    # Call GPT to perform formal_s to natural language
    
    prompt = f"""
            Translate the following natural language statement into 10 clear Isabelle/HOL statements that conveys the exact same logical meaning.
            
            Generate a numbered list of 10 unique Isabelle/HOL statements, without any additional commentary or explanation.

            Natural Language Statement:
            {natural_s}

            Formal Statements:
            1.
            """
    
    # Call GPT
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that translates between formal logic and natural language."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=5000,
        n=1,
        stop=None,
        temperature=0.0,
    )

    # Collect Response
    gpt_formal_s_2 = response.choices[0].message.content

    # keep the entire entry but add gpt_formal_s_2
    results[entry] = {
        "natural_s": natural_s,
        "natural_p": natural_p,
        "formal_s": formal_s,
        "gpt_formal_s_2": gpt_formal_s_2
    }

# Save results to a JSON file
with open('gpt_translations_latex_10_2.json', 'w') as out_file:
    json.dump(results, out_file, indent=4)

Processing entries: 100%|██████████| 244/244 [22:44<00:00,  5.59s/it]


In [ ]:
with open('gpt_translations_latex_10_2.json', 'r') as out_file:
    my_json = json.load(out_file)


for entry in my_json:
    # Try to split based on "\item" after removing "\begin{enumerate}" and "\end{enumerate}"
    # Otherwise we need to split based on new lines
    
    items = my_json[entry]["gpt_formal_s"].split("\n")
    # Remove leading numbers after splitting on dot or parenthesis
    items = [item.split('.', 1)[-1] if '.' in item else item.split(')', 1)[-1] for item in items]

    # Add "theorem:\n " at the start of all if not present
    # Loop through items and add "theorem:\n " if not present
    for i in range(len(items)):
        if not items[i].strip().startswith("theorem:"):
            items[i] = "theorem:\n " + items[i].strip()
        else:
            items[i] = items[i].strip()
            
    items = [item.strip() for item in items if item.strip()]


    my_json[entry]["gpt_formal_s_list"] = items
    
   
with open('gpt_translations_latex_10_formatted_2.json', 'w') as out_file:
    json.dump(my_json, out_file, indent=4)


In [10]:
from isabelle_client import get_isabelle_client

isa = get_isabelle_client(
    'server "myserver" = 127.0.0.1:62502 (password "96599a52-6f88-4d50-a3b6-4a3d93033567")'
)


In [13]:
import os, textwrap, concurrent.futures

# (Optional) ensure Scratch.thy exists in the current dir
if not os.path.exists("Scratch.thy"):
    with open("Scratch.thy", "w", encoding="utf-8") as f:
        f.write(textwrap.dedent("""\
            theory Scratch
            imports Main
            begin

            lemma "∀x. x = x" by simp

            end
        """))

from isabelle_client import get_isabelle_client

isa = get_isabelle_client(
    'server "myserver" = 127.0.0.1:62502 (password "96599a52-6f88-4d50-a3b6-4a3d93033567")'
)

def blocking(fn, /, *args, **kwargs):
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(lambda: fn(*args, **kwargs)).result()

def get_session_id(x):
    # some versions return a dict, others a plain string
    return x.get("session_id") if isinstance(x, dict) else x

sid = get_session_id(blocking(isa.session_start, session="HOL"))

res = blocking(
    isa.use_theories,
    session_id=sid,
    theories=["Scratch"],
    master_dir=".",          # directory containing Scratch.thy
    unicode_symbols=True,
    watchdog_timeout=0
)

print(res.get("ok"), res.get("errors", []))
blocking(isa.shutdown)


AttributeError: 'list' object has no attribute 'get'